In [ ]:
import pandas as pd
import string
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
nlp = spacy.load("en_core_web_sm", exclude=["parser", "ner"])

In [ ]:
import json

try:
    # Read the file line by line and parse each line as a JSON object
    data = []
    with open('/content/News_Category_Dataset_v3.json', 'r') as f:
      # itera su ogni riga del file
        for line in f:
            try:
              # converte la riga corrente in un dizionario
                data.append(json.loads(line))
            except json.JSONDecodeError as e:
                print(f"Skipping malformed JSON line: {line.strip()} - Error: {e}")
    df = pd.DataFrame(data)

except FileNotFoundError:
    print("Error: Please download the dataset from Kaggle and place it in the directory.")
    exit()

Skipping malformed JSON line: {"category": "ENTERTAINMENT", "headline": "'You Raped Women, Bill Cosby': How A Bit In Hannibal Buress's Standup Led To The Takedown Of An Icon", "authors": "Suzy Byrne, Yahoo Entertainment", - Error: Expecting property name enclosed in double quotes: line 1 column 193 (char 192)


In [ ]:
# Random subset of 10,000 headlines
subset_df = df.sample(n=10000, random_state=42).reset_index(drop=True)      # reset_index, resetta gli indici delle righe in modo che vadano da 0 a 9999

# estrae la colonna contenente i titoli delle notizie trasformando in una lista
headlines = subset_df['headline'].tolist()

def preprocess_query(text):
    # il testo viene tokenizzato
    doc = nlp(text)

    processed_tokens = [
        # si trasforma ogni parola nella forma base/lemma e la si converte in minuscolo
        str(token.lemma_).lower()
        for token in doc
        # vengono escluse le stop words e la punteggiatura
        if not token.is_stop and not token.is_punct
    ]
    return " ".join(processed_tokens)

In [ ]:
print("Preprocessing text in batch")
preprocessed_docs = []

# Sfruttiamo nlp.pipe per efficienza sui 10.000 documenti
for doc in nlp.pipe(headlines, batch_size=256):
    processed_tokens = [
        str(token.lemma_).lower()
        for token in doc
        if not token.is_stop and not token.is_punct
    ]
    preprocessed_docs.append(" ".join(processed_tokens))

# Vector Space Model Representation
# viene calcolata l'importanza di ogni parola all'interno di cascun titolo
vectorizer = TfidfVectorizer()

# Convertiamo i documenti preprocessati in TF-IDF matrix
# fit - trasforma tutti i documenti per creare il vocabolario delle parole 
# transform i 10000 titoli vengono trasformati in una matrice sparsa, ogni riga è un titolo,
#   ogni colonna rappresenta il punteggio TF-IDF di una parola
tfidf_matrix = vectorizer.fit_transform(preprocessed_docs)

# Salviamo il vocabolario sul vectorizer
print(f"Vocabulary size: {len(vectorizer.vocabulary_)} words")

Preprocessing text in batch
Vocabulary size: 13125 words


In [ ]:
# cerca e restituisce i documenti più rilevanti in base alla query
def retrieve_documents(query, vectorizer, tfidf_matrix, original_docs, top_n=5):

    # viene processata la query -> lemmatizzate, minuscola, rimosse stop-words
    preprocessed_query = preprocess_query(query)

    # Convert TF-IDF vector
    query_vector = vectorizer.transform([preprocessed_query])

    # calcolo della cosine similarity tra la query e i documenti
    similarities = cosine_similarity(query_vector, tfidf_matrix).flatten()

    # Indici di similarità della top N valori
    # - .argsort() ordina gli indici in base al punteggio in ordine crescente
    # - [-top_n:] prende gli ultimi 'top_n' elementi (i punteggi più alti)
    # - [::-1] inverte l'ordine, mettendo il punteggio più alto in cima alla classifica
    top_indices = similarities.argsort()[-top_n:][::-1]

    print(f"\nTop {top_n} Results for: '{query}'")
    for i, idx in enumerate(top_indices):
        score = similarities[idx]
        print(f"{i+1}. [Score: {score:.4f}] {original_docs[idx]}")

    return top_indices

if __name__ == "__main__":
    print("\nDocument Retrieval System Ready")
    while True:
        user_query = input("\nEnter a query (or type 'exit' to quit): ")
        if user_query.lower() == 'exit':
            break
        if not user_query.strip():
            continue

        retrieve_documents(user_query, vectorizer, tfidf_matrix, headlines)


Document Retrieval System Ready

Enter a query (or type 'exit' to quit): basketball championship finals

Top 5 Results for: 'basketball championship finals'
1. [Score: 0.2773] British Woman Conquers World Conkers Championship With Only 20 Minutes Of Training
2. [Score: 0.2335] Florida Basketball Player Who Collapsed During Game Is Released From Hospital
3. [Score: 0.2187] Kobe Bryant One Of 8 Finalists For Basketball Hall Of Fame
4. [Score: 0.2165] John Lewis’s Final Fight For Voting Rights
5. [Score: 0.2089] Virginia State University President Takes Student Back To School In Viral Basketball Clip

Enter a query (or type 'exit' to quit): government budget deficit

Top 5 Results for: 'government budget deficit'
1. [Score: 0.3063] Trump Hits A New High-Water Mark: The Biggest Federal Deficit In 6 Years
2. [Score: 0.2933] GOP No Longer Cares About The Deficit If It Means Women Having More Babies
3. [Score: 0.2574] Democrats Consider Abandoning Budget Floor Vote
4. [Score: 0.2488] Congres

# Query di esempio:
- "government budget deficit" (Politics)
- "apple smartphone release" (Tech)
- "basketball championship finals" (Sports)
- "climate change emissions" (Environment)
- "oscar awards red carpet" (Entertainment)
- "stock market crash" (Business)
- "healthy diet recipes" (Wellness/Food)
- "supreme court ruling" (Politics/U.S. News)
- "artificial intelligence breakthrough" (Tech/Science)
- "travel restrictions europe" (Travel)



# Analisi dei Risultati: Document Similarity e Limiti del Modello TF-IDF

Notando i risultati finali, possiamo trarre conclusioni sui limiti del Vector Space Model (VSM) basato su embedding sparsi (TF-IDF).

## 1. Rilevanza dei risultati (Cosa funziona bene)
Nella maggior parte dei casi, i risultati appaiono **altamente rilevanti** quando la query è composta da termini che condividono un forte legame lessicale con il dominio di destinazione.
* Ad esempio, per query come *"supreme court ruling"*, *"stock market crash"* e *"government budget deficit"*, il sistema restituisce notizie perfettamente centrate (es. articoli sulla Corte Suprema, sull'andamento dei mercati o sul deficit federale).
* Questo dimostra la forza della metrica **TF-IDF (Term Frequency-Inverse Document Frequency)**: penalizza parole comuni a molti documenti (come articoli o preposizioni, se non già rimosse) e premia termini specifici e discriminanti (come "deficit" o "ruling"), permettendo ai documenti che li contengono di ottenere uno *score* di similarità del coseno molto alto.

## 2. Falsi positivi e Limiti Semantici (I problemi del TF-IDF)
Analizzando query più complesse, emergono chiaramente i limiti di questo approccio *Bag-of-Words*:

* **Mancanza di comprensione contestuale (N-grammi non preservati):** Il caso più evidente è la query **"artificial intelligence breakthrough"**. Il primo risultato restituito è *"Artificial Christmas Trees: Here's What You Should Know This Year"* (Score: 0.4402), seguito da articoli politici sul *"Senate Intelligence Committee"*.
Questo succede perchè il modello elabora i token individualmente. La parola "artificial" ha probabilmente un peso IDF molto alto (essendo rara nel corpus) e fa salire in cima l'articolo sull'albero di Natale. La parola "intelligence" viene associata ai servizi segreti. Il sistema non ha modo di sapere che "artificial intelligence" rappresenta un singolo concetto semantico, poiché la struttura sintattica e l'ordine delle parole vengono persi.

* **Matching lessicale vs. Intento semantico:** Nella query **"apple smartphone release"**, i risultati sono saturati da notizie sul brand Apple (Apple TV, AirPods, Apple Store), ma nessuno parla strettamente di "smartphone" o di "release". Questo accade perché il TF-IDF richiede il *matching esatto* della stringa. Se un documento pertinente usasse i termini "iPhone", "debut" o "announcement" ma non specificamente "apple", "smartphone" e "release", il sistema non riuscirebbe a recuperarlo, ignorando completamente i sinonimi.

## Conclusioni
Il modello basato su TF-IDF si conferma un'ottima *baseline*: è computazionalmente leggero, veloce e molto preciso per ricerche *keyword-based*. Tuttavia, per superare i problemi di polisemia (stessa parola, significati diversi) e la mancanza di composizionalità semantica evidenziata nei risultati, sarebbe necessario passare a modelli di embedding densi (come Word2Vec o architetture basate su Transformer), capaci di catturare il contesto e il significato latente delle frasi.